### Wrap the SQL side as a genuinely reusable Unity Catalog function

In [0]:
%sql
CREATE OR REPLACE FUNCTION policyiq.gold.get_compliance_status(
  p_branch_id STRING DEFAULT NULL,
  p_policy_domain STRING DEFAULT NULL,
  p_compliance_filter STRING DEFAULT 'ALL'  -- 'ALL' | 'NON_COMPLIANT' | 'COMPLIANT'
)
RETURNS TABLE (
  branch_id STRING, branch_name STRING, policy_domain STRING, kpi_name STRING,
  kpi_display_name STRING, policy_id STRING, policy_section_ref STRING,
  actual_value DOUBLE, threshold_value DOUBLE, threshold_operator STRING,
  unit STRING, compliance_status STRING, raw_gap DOUBLE
)
COMMENT 'Returns branch-level KPI compliance results with the exact policy clause reference for each row. Use this to answer any question about which branches violate a policy, current compliance status, or gap-to-threshold.'
RETURN
  SELECT branch_id, branch_name, policy_domain, kpi_name, kpi_display_name,
         policy_id, policy_section_ref, actual_value, threshold_value,
         threshold_operator, unit, compliance_status, raw_gap
  FROM policyiq.silver.kpi_compliance_facts
  WHERE (p_branch_id IS NULL OR branch_id = p_branch_id)
    AND (p_policy_domain IS NULL OR policy_domain = p_policy_domain)
    AND (p_compliance_filter = 'ALL'
         OR (p_compliance_filter = 'NON_COMPLIANT' AND compliance_status = 'Non-Compliant')
         OR (p_compliance_filter = 'COMPLIANT' AND compliance_status = 'Compliant'));

In [0]:
%sql
SELECT * FROM policyiq.gold.get_compliance_status(
  p_branch_id => NULL,
  p_policy_domain => 'aml_mltf',
  p_compliance_filter => 'NON_COMPLIANT'
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION policyiq.gold.search_policy_text(
  p_query STRING,
  p_policy_domain STRING DEFAULT NULL
)
RETURNS TABLE (policy_id STRING, policy_name STRING, policy_section_ref STRING, chunk_text STRING)
COMMENT 'Semantic search over all policy documents. Use this to answer questions about what a policy requires, defines, or states, when you need the exact clause text rather than a compliance number.'
RETURN
  SELECT policy_id, policy_name, NULL AS policy_section_ref, chunk_text
  FROM VECTOR_SEARCH(
    index => 'policyiq.silver.policy_chunks_index',
    query => p_query,
    num_results => 5
  )
  WHERE p_policy_domain IS NULL OR policy_domain = p_policy_domain;

In [0]:
%sql
SELECT * FROM policyiq.gold.search_policy_text(
  p_query => 'related party lending limit as percentage of capital',
  p_policy_domain => 'credit_risk'
);